In [2]:
import plotly.graph_objects as go

# =========================================
# INPUTS
# =========================================
text_to_display = "कुल्लर वी लगा रख्खा"
bg_color = "rgb(245, 245, 245)"
font_size = 20  # Reduced font size
output_filename = "devanagari_small.png"

# =========================================
# GENERATE FIGURE
# =========================================
fig = go.Figure()

fig.add_annotation(
    x=0.5,
    y=0.5,
    text=text_to_display,
    showarrow=False,
    xref="paper",
    yref="paper",
    font=dict(
        family="Arial, sans-serif", size=font_size, color="black"  # Applied here
    ),
    align="center",
)

fig.update_layout(
    width=800,
    height=150,  # Reduced height slightly to match smaller text
    plot_bgcolor=bg_color,
    paper_bgcolor=bg_color,
    xaxis=dict(showgrid=False, zeroline=False, visible=False, range=[0, 1]),
    yaxis=dict(showgrid=False, zeroline=False, visible=False, range=[0, 1]),
    margin=dict(l=0, r=0, t=0, b=0),
)

fig.show()

# To save for your paper:
fig.write_image(output_filename, scale=2)

In [ ]:
FIXED_UTT = [
    "arctic_a0094",
    "arctic_a0220",
    "arctic_a0346",
    "arctic_a0472",
    "arctic_b0005",
    "arctic_b0131",
    "arctic_b0257",
    "arctic_b0383",
    "arctic_b0509",
    "arctic_b0258",
    "arctic_b0384",
    "arctic_b0510",
    # "arctic_a0095",
    # "arctic_a0221",
    # "arctic_a0347",
    # "arctic_a0473",
    # "arctic_b0006",
    # "arctic_b0132",
    # "arctic_a0096",
    # "arctic_a0222",
    # "arctic_a0348",
    # "arctic_a0474",
    # "arctic_b0007",
    # "arctic_b0133",
]
ORIGINAL_PATH = "/data/user_data/sbharad2/PhoneBench/exp/runs/inf_cmul2arcticl1_ctag/20251213_181200/transcription.json"
with open(ORIGINAL_PATH, "r") as f:
    references = json.load(f)

ctr = 0
smaller_set_transcriptions = {}
for k in references.keys():
    if references[k]["passthrough"]["utt_id"] in FIXED_UTT:
        smaller_set_transcriptions[k] = references[k]
        ctr += 1
print(ctr)
savepath = "/data/user_data/sbharad2/PhoneBench/exp/runs/inf_cmul2arcticl1_ctag/20251213_181200/transcription.small.json"
with open(savepath, "w") as f:
    json.dump(smaller_set_transcriptions, f, ensure_ascii=False, indent=4)

329


In [57]:
import os
import sys
import torch
import hydra
from hydra import initialize, compose
from hydra.core.global_hydra import GlobalHydra
import json
import pandas as pd

sys.path.append("/data/user_data/sbharad2/PhoneBench")
from src.core.tools.attribution import ModelInterpreter
from src.core.tools.visualizer import (
    visualize_text_interactive,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [58]:
if GlobalHydra.instance().is_initialized():
    GlobalHydra.instance().clear()

# CKPT_PATH="/data/user_data/sbharad2/PhoneBench/exp/runs/cascade.easycall_lv60/20251230_124940/checkpoints/step_001906.ckpt"
# JSON_PATH="/data/user_data/sbharad2/PhoneBench/exp/runs/inf_easycall_lv60/8jobARR/transcription.json"

# CKPT_PATH="/data/user_data/sbharad2/PhoneBench/exp/runs/cascade.uaspeech_powsm_ctc/20251228_202118/checkpoints/step_002412.ckpt"
# JSON_PATH="/data/user_data/sbharad2/PhoneBench/exp/runs/inf_uaspeech_powsm_ctc/8jobARR/transcription.json"

# ZIPANS
# CKPT_PATH = "/data/user_data/sbharad2/PhoneBench/exp/runs/cascade.geo_in_zipactc_ns/20251228_173247/checkpoints/step_021078.ckpt"
# JSON_PATH = "/data/user_data/sbharad2/PhoneBench/exp/runs/inf_vaanigeo_zipactc_ns/8jobARR/transcription.json"
# VOCAB_SIZE = 86  # 92

# Vaani-lv60
# CKPT_PATH = "/data/user_data/sbharad2/PhoneBench/exp/runs/cascade.geo_in_lv60/20251228_173318/checkpoints/step_015034.ckpt"
# JSON_PATH = "/data/user_data/sbharad2/PhoneBench/exp/runs/inf_vaanigeo_lv60/8jobARR/transcription.json"
# VOCAB_SIZE = 104

# CMU L2 Arctic
CKPT_PATH = "/data/user_data/sbharad2/PhoneBench/exp/runs/cascade.cmul2arctic_ctag/20251221_200117/checkpoints/step_003074.ckpt"
# JSON_PATH = "/data/user_data/sbharad2/PhoneBench/exp/runs/inf_cmul2arcticl1_ctag/20251213_181200/transcription.json"
JSON_PATH = "/data/user_data/sbharad2/PhoneBench/exp/runs/inf_cmul2arcticl1_ctag/20251213_181200/transcription.small.json"
METADATA_PATH = (
    "/data/user_data/sbharad2/PhoneBench/exp/cache/cmu_l2arctic/metadata.csv"
)
# METADATA_PATH = (
#     "/data/user_data/sbharad2/PhoneBench/exp/cache/cmu_l2arctic/metadata.small.csv"
# )
VOCAB_SIZE = 90

NUM_CLASSES = 7  # 5
with initialize(version_base=None, config_path="../../../configs"):
    cfg = compose(
        config_name="main",
        overrides=[
            # "experiment=cascade/rnn_geolocation.yaml",
            "experiment=cascade/rnn_classification.yaml",
            f"ckpt_path={CKPT_PATH}",
            f"data.json_path={JSON_PATH}",
            f"data.num_classes={NUM_CLASSES}",
            f"data.batch_size=1",
            f"model.net.vocab_size={VOCAB_SIZE}",
        ],
    )

print(f"Loading Model: {cfg.model._target_}")

# Instantiate
datamodule = hydra.utils.instantiate(cfg.data)
datamodule.prepare_data()
datamodule.setup()

model = hydra.utils.instantiate(cfg.model)

if cfg.get("ckpt_path"):
    print(f"Loading weights from {cfg.ckpt_path}")
    checkpoint = torch.load(cfg.ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["state_dict"])

model.to(device)
# model.train()
model.eval()

with open(JSON_PATH, "r") as f:
    references = json.load(f)
metadata_df = pd.read_csv(METADATA_PATH)

Loading Model: src.recipe.common.classification_module.ClassificationModel
Loading weights from /data/user_data/sbharad2/PhoneBench/exp/runs/cascade.cmul2arctic_ctag/20251221_200117/checkpoints/step_003074.ckpt


In [59]:
def explain_prediction(batch, model, interpreter, cfg, datamodule):
    assert cfg.model.input_type == "ipa", "This script only supports text input type."
    print("\n=== Running Text Integrated Gradients ===")
    assert hasattr(datamodule, "tokenizer"), "Tokenizer required for text IG."
    results = interpreter.integrated_gradients_text(
        batch["text"],
        batch["lengths"],
        batch["target"],
        datamodule.tokenizer,
        steps=200,
    )
    return results
    # df_agg = interpreter.aggregate_by_phonetic_class(results)
    # return df_agg


def plot_audio(pth):
    import torchaudio
    import IPython.display as ipd

    waveform, sample_rate = torchaudio.load(pth)
    ipd.display(ipd.Audio(waveform.numpy(), rate=sample_rate))

In [60]:
metadata_df

,audio_path,l1_label,split,speaker_id,utt_id
0,l2arctic/ABA/wav/arctic_b0114.wav,ar,train,ABA,arctic_b0114
1,l2arctic/ABA/wav/arctic_b0406.wav,ar,train,ABA,arctic_b0406
2,l2arctic/ABA/wav/arctic_a0224.wav,ar,train,ABA,arctic_a0224
3,l2arctic/ABA/wav/arctic_a0579.wav,ar,train,ABA,arctic_a0579
4,l2arctic/ABA/wav/arctic_b0163.wav,ar,train,ABA,arctic_b0163
...,...,...,...,...,...
31390,cmu/cmu_us_rms_arctic/wav/arctic_a0290.wav,en,train,RMS,arctic_a0290
31391,cmu/cmu_us_rms_arctic/wav/arctic_a0472.wav,en,train,RMS,arctic_a0472
31392,cmu/cmu_us_rms_arctic/wav/arctic_b0340.wav,en,train,RMS,arctic_b0340
31393,cmu/cmu_us_rms_arctic/wav/arctic_b0182.wav,en,train,RMS,arctic_b0182


In [36]:
batch = next(iter(datamodule.val_dataloader()))
print(batch)

p = get_l2arc_path(batch["metadata_idx"][0])
plot_audio(p)

{'text': tensor([[19, 39,  5,  4,  5, 39, 16,  5, 16, 22, 24, 22, 13, 37, 16,  6, 19, 39,
         16,  6,  4, 20, 22]]), 'lengths': tensor([23]), 'target': tensor([1.]), 'utt_id': ['arctic_a0220'], 'sample_index': ['22504'], 'metadata_idx': ['29134']}


In [61]:
def get_vaani_path(utt_id):
    return f"/data/user_data/sbharad2/PhoneBench/exp/cache/vaanihindi/saved_val/{utt_id}.wav"


def get_l2arc_path(metadataidx):
    metadataidx = int(metadataidx)
    row = metadata_df.iloc[metadataidx]
    print(row)
    path = row["audio_path"]
    PREFIX = "/data/user_data/sbharad2/PhoneBench/exp/download/cmu_l2arctic/resampled_16000Hz"
    return f"{PREFIX}/{path}"


uttids = []
metaids = []
all_results = []
interpreter = ModelInterpreter(model)
for i, batch in enumerate(datamodule.val_dataloader()):
    # if i > 10:
    # break
    batch = {
        k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()
    }
    batch_results = interpreter.integrated_gradients_text(
        batch["text"],
        batch["lengths"],
        batch["target"],
        datamodule.tokenizer,
        steps=200,
    )
    uttids.extend(batch["utt_id"])
    metaids.extend(batch["metadata_idx"])
    all_results.extend(batch_results)
    if i % 100 == 0:
        print(f"Processed {i} batches.")

Processed 0 batches.


In [65]:
len(uttids), len(set(uttids)), len(metaids)

(83, 12, 83)

In [66]:
batch_results = []
for ar in all_results:
    batch_results.append(ar)

In [67]:
len(batch_results)

83

In [ ]:
for i in range(len(batch_results)):
    if uttids[i] != "arctic_b0131":
        continue
    print(uttids[i])
    visualize_text_interactive(batch_results[i])
    plot_audio(get_l2arc_path(metaids[i]))
    print("===" * 20)

arctic_b0131


audio_path    cmu/cmu_us_clb_arctic/wav/arctic_b0131.wav
l1_label                                              en
split                                                val
speaker_id                                           CLB
utt_id                                      arctic_b0131
Name: 29450, dtype: object


arctic_b0131


audio_path    l2arctic/YBAA/wav/arctic_b0131.wav
l1_label                                      ar
split                                        val
speaker_id                                  YBAA
utt_id                              arctic_b0131
Name: 2444, dtype: object


arctic_b0131


...

audio_path    l2arctic/YDCK/wav/arctic_b0131.wav
l1_label                                      ko
split                                        val
speaker_id                                  YDCK
utt_id                              arctic_b0131
Name: 16493, dtype: object


arctic_b0131


audio_path    l2arctic/ERMS/wav/arctic_b0131.wav
l1_label                                      es
split                                        val
speaker_id                                  ERMS
utt_id                              arctic_b0131
Name: 19738, dtype: object


arctic_b0131


audio_path    l2arctic/PNV/wav/arctic_b0131.wav
l1_label                                     vi
split                                       val
speaker_id                                  PNV
utt_id                             arctic_b0131
Name: 23865, dtype: object


In [ ]:
for i in range(len(batch_results)):
    print(uttids[i])
    visualize_text_interactive(batch_results[i])
    plot_audio(get_vaani_path(uttids[i]))
    print("===" * 20)

val_0


val_1


val_2


val_3


val_4


...

val_1835


val_1836


val_1837


val_1838


val_1839


In [5]:
interpreter = ModelInterpreter(model)
i = 0
# for batch in datamodule.val_dataloader():
batch = next(iter(datamodule.val_dataloader()))
batch = {
    k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()
}
# pred = model.model_step(batch)
# print(pred)
result = explain_prediction(batch, model, interpreter, cfg, datamodule)
# for i in range(len(result["attributions"])):
#     print("Sample", i)
#     print("Attributions:", result["attributions"][i])
#     print("Text:", result["text"][i])
#     print("Target:", result["target"][i])
#     sample_result = {}
#     for key in result:
#         sample_result[key] = result[key][i]
#     visualize_text_interactive(sample_result)
#     print("Predicted:", result["predicted"][i])
#     uttid = batch["uttid"][i]
#     pth = f"/data/user_data/sbharad2/PhoneBench/exp/cache/vaanihindi/saved_val/{uttid}"
#     print("Audio Path:", pth)
#     plot_audio(pth)
#     print("==" * 10)
#     i += 1
#     if i >= 2:
#         break
#     break


=== Running Text Integrated Gradients ===


In [ ]:
batch["target"].shape

torch.Size([64])

In [ ]:
model.model_step(batch)

{'loss': tensor(0.1751, grad_fn=<MseLossBackward0>),
 'preds': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0]),
 'targets': tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]),
 'logits': tensor([[-0.4207],
         [-0.4077],
         [ 0.0931],
         [-0.0145],
         [-0.2812],
         [-0.3479],
         [ 0.3305],
         [-0.0599],
         [-0.3606],
         [-0.0540],
         [-0.2154],
         [-0.8492],
         [-0.2618],
         [-0.0604],
         [ 0.0853],
         [-0.9939],
         [-0.2874],
         [-0.3293],
         [-0.4559],
         [-0.0248],
         [-

In [ ]:
# Force clear hooks from the embedding layer
# (Adjust the path 'interpreter.model.net.embedding' if your model structure is different)
for name, module in model.named_modules():
    if hasattr(module, "_forward_hooks"):
        module._forward_hooks.clear()
print("hook cleared")

All hooks cleared. You can now run the analysis.


In [ ]:
RESULTS = """Model	ModelString	Setup	Dataset	Macro F1	F1 of macroP&R	precision	recall
anyspeech/zipa-large-crctc-500k	ZIPA-CTC	Cascade	Doreco-45	0.552	0.560	0.422	0.831
anyspeech/zipa-large-crctc-ns-800k	ZIPA-CTC-NS	Cascade	Doreco-45	0.566	0.573	0.441	0.817
espnet/powsm	POWSM	Cascade	Doreco-45	0.487	0.493	0.364	0.763
na	POWSM-CTC	Cascade	Doreco-45	0.577	0.586	0.495	0.717
ctaguchi/wav2vec2-large-xlsr-japlmthufielta-ipa1000-ns	MultiIPA	Cascade	Doreco-45	0.409	0.414	0.278	0.813
facebook/wav2vec2-lv-60-espeak-cv-ft	W2V2Ph-LV60	Cascade	Doreco-45	0.513	0.519	0.382	0.811
facebook/wav2vec2-xlsr-53-espeak-cv-ft	W2V2Ph-XLSR53	Cascade	Doreco-45	0.569	0.461	0.461	0.461
na	Gemini 2.5 Flash	Cascade	Doreco-45	0.391	0.399	0.256	0.899
na	Qwen3-Omni	Cascade	Doreco-45	0.445	0.452	0.305	0.875"""

In [ ]:
import pandas as pd
import numpy as np
from io import StringIO
import plotly.graph_objects as go

# Adjusted YRANGE start to 0.25 to include Gemini (0.256)
XRANGE = [0.42, 0.92]
YRANGE = [0.25, 0.52]

# 1. Load Data
df = pd.read_csv(StringIO(RESULTS), sep="\t")

# 2. Generate Grid
x = np.linspace(XRANGE[0], XRANGE[1], 200)
y = np.linspace(YRANGE[0], YRANGE[1], 200)
X, Y = np.meshgrid(x, y)
Z = 2 * (X * Y) / (X + Y)

fig = go.Figure()

# 3. Add F1 Contours
fig.add_trace(
    go.Contour(
        x=x,
        y=y,
        z=Z,
        showscale=False,
        hoverinfo="skip",
        colorscale=[[0, "#ACA1A1"], [1, "#ACA1A1"]],
        contours=dict(
            coloring="lines",
            showlabels=True,
            start=0.40,
            end=0.60,
            size=0.05,
            labelfont=dict(size=10, color="#ACA1A1"),
        ),
        line=dict(width=1, dash="dash"),
    )
)

# 4. Add Model Points
fig.add_trace(
    go.Scatter(
        x=df["recall"],
        y=df["precision"],
        mode="markers",
        marker=dict(
            size=10, color="#636EFA", line=dict(width=1, color="DarkSlateGrey")
        ),
        text=df["ModelString"],
        customdata=df["F1 of macroP&R"],
        hovertemplate="<b>%{text}</b><br>R: %{x:.3f}<br>P: %{y:.3f}<br>F1: %{customdata:.3f}<extra></extra>",
    )
)

# 5. Add Labels
for _, row in df.iterrows():
    is_left = row["ModelString"] == "W2V2Ph-XLSR53"
    push_left = 0.01 if is_left else -0.01
    push_down = -0.005 if row["ModelString"] == "Qwen3-Omni" else 0.0

    fig.add_annotation(
        x=row["recall"] + push_left,
        y=row["precision"] + push_down,
        text=row["ModelString"],
        showarrow=False,
        xanchor="left" if is_left else "right",
        font=dict(size=11),
    )

# 6. Layout
fig.update_layout(
    width=500,
    height=450,
    template="plotly_white",
    # EXTREMELY TIGHT MARGINS to save paper space
    # l/b need just enough room for axis text; r/t can be near zero
    margin=dict(l=40, r=5, t=5, b=35),
    xaxis=dict(
        title=dict(text="Recall", standoff=0),  # Standoff=0 touches the ticks
        range=XRANGE,
        showline=True,
        mirror=True,
        linecolor="black",
    ),
    yaxis=dict(
        title=dict(text="Precision", standoff=0),
        range=YRANGE,
        showline=True,
        mirror=True,
        linecolor="black",
    ),
)

fig.write_image("pr-scatter.pdf")
fig.show()

In [ ]:
import json
import unicodedata
import sys

sys.path.append("/data/user_data/sbharad2/PhoneBench")
from src.model.sentencepieces_tokenizer import SentencepiecesTokenizer

# 1. Load the vocabularies (Assuming passed as strings for this demo)
xlsrf_vocab_pth = "/data/user_data/sbharad2/PhoneBench/src/recipe/l1_classification/local/vocabxlsrf.json"

tokenizer = SentencepiecesTokenizer(
    model="/data/user_data/sbharad2/PhoneBench/src/model/zipa/resources/unigram_127.model"
)
tokenizer._build_sentence_piece_processor()
zipa_vocab = {}
for i in range(tokenizer.sp.GetPieceSize()):
    piece = tokenizer.sp.IdToPiece(i)
    zipa_vocab[piece] = i

vocab_a = json.load(open(xlsrf_vocab_pth, "r"))
vocab_b = zipa_vocab


def analyze_vocab(vocab, name):
    stats = {
        "total_size": len(vocab),
        "multi_char_tokens": 0,
        "combining_modifiers": 0,
        "base_tokens": 0,
        "tone_numbers": 0,
    }

    examples = {"multi": [], "modifier": []}

    for token, idx in vocab.items():
        if token.startswith("<") or token in ["[PAD]", " "]:
            continue

        # Check for tone numbers (common in Vocab A)
        if any(char.isdigit() for char in token):
            stats["tone_numbers"] += 1

        # Check for combining characters (The hallmark of Vocab B)
        # Categories 'Mn', 'Mc', 'Me' are non-spacing marks (modifiers)
        if len(token) == 1 and unicodedata.category(token).startswith("M"):
            stats["combining_modifiers"] += 1
            examples["modifier"].append(token)

        # Check for pre-composed clusters (The hallmark of Vocab A)
        elif len(token) > 1:
            stats["multi_char_tokens"] += 1
            if len(examples["multi"]) < 5:
                examples["multi"].append(token)
        else:
            stats["base_tokens"] += 1

    print(f"--- Analysis of {name} ---")
    print(f"Total Size: {stats['total_size']}")
    print(f"Pre-composed Clusters (e.g., 'tʃ', 'nʲ'): {stats['multi_char_tokens']}")
    print(f"Standalone Modifiers (e.g., 'ʲ', '̃'): {stats['combining_modifiers']}")
    print(f"Tokens with Tone Numbers: {stats['tone_numbers']}")
    print(f"Examples (Multi-char): {examples['multi']}")
    print(f"Examples (Modifiers): {examples['modifier']}")
    print("\n")


analyze_vocab(vocab_a, "Vocab A (Accurate)")
analyze_vocab(vocab_b, "Vocab B (Inaccurate)")

--- Analysis of Vocab A (Accurate) ---
Total Size: 392
Pre-composed Clusters (e.g., 'tʃ', 'nʲ'): 302
Standalone Modifiers (e.g., 'ʲ', '̃'): 0
Tokens with Tone Numbers: 85
Examples (Multi-char): ['iː', 'aɪ', 'ɑː', 'eɪ', 'uː']
Examples (Modifiers): []


--- Analysis of Vocab B (Inaccurate) ---
Total Size: 127
Pre-composed Clusters (e.g., 'tʃ', 'nʲ'): 0
Standalone Modifiers (e.g., 'ʲ', '̃'): 7
Tokens with Tone Numbers: 0
Examples (Multi-char): []
Examples (Modifiers): ['̃', '̚', '̥', '̩', '̪', '̴', '̺']




In [4]:
import pandas as pd
import plotly.graph_objects as go

df = pd.read_csv("geolocation_ig_phonetic_class_attribution.csv")

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np


def plot_phonetic_attribution_pretty(df: pd.DataFrame):
    # 1. Data Prep
    stats = (
        df.groupby("Category")["Attribution"]
        .agg(["mean", "count", "sem"])
        .reset_index()
    )
    stats["ci95"] = stats["sem"] * 1.96
    stats = stats.sort_values("mean", ascending=False)

    # 2. Build Plotly Figure
    fig = go.Figure(
        go.Bar(
            x=stats["Category"],
            y=stats["mean"],
            error_y=dict(
                type="data",
                array=stats["ci95"],
                visible=True,
                color="#333333",
                thickness=1.5,
                width=6,
            ),
            marker=dict(
                color="#4C72B0",
                opacity=0.9,
                cornerradius=10,
            ),
            hovertemplate="<b>%{x}</b><br>Mean: %{y:.4f}<br>Count: %{customdata} samples<extra></extra>",
            customdata=stats["count"],
        )
    )

    # 3. Add "n=..." Annotations
    annotations = []
    for index, row in stats.iterrows():
        y_pos = row["mean"] + row["ci95"]
        annotations.append(
            dict(
                x=row["Category"],
                y=y_pos,
                text=f"n={int(row['count'])}",
                showarrow=False,
                yshift=12,
                font=dict(size=11, color="#555555"),
            )
        )

    # 4. Final Polish Layout (Zero Margins)
    fig.update_layout(
        yaxis=dict(
            title="<b>Mean Attribution Mass</b> (95% CI)",
            gridcolor="#EAEAEA",
            zeroline=False,
            tickfont=dict(size=12),
        ),
        xaxis=dict(
            title="",
            tickfont=dict(size=13),
        ),
        template="plotly_white",
        annotations=annotations,
        width=700,
        height=500,
        # --- CHANGES HERE ---
        # Set all margins to 0 to eliminate whitespace
        margin=dict(l=0, r=0, t=0, b=0),
        # --------------------
        showlegend=False,
    )

    # Note: When saving to PDF with 0 margins, the axis labels might get clipped
    # if the 'width'/'height' are too tight for the content.
    # Usually, a very small margin (l=50, b=50) is needed for axis titles,
    # but strictly l=0, r=0, t=0, b=0 will do what you asked.

    fig.write_image("phone-attribution.pdf")
    fig.show()


plot_phonetic_attribution_pretty(df)